# Notebook 10-B COLAB — BETO Mejorado: Hiperparametros de Investigacion (F1 ~0.93)
### Entorno: Google Colab — Runtime T4 GPU | Con checkpoints reanudables

---

## Objetivo
Replicar la configuracion de hiperparametros publicada por investigadores que
alcanzaron **F1-Score 0.931** con BETO, adaptandola a nuestro corpus y benchmark.

### Configuracion de los investigadores (referencia)

| Hiperparametro | Valor |
|---|---|
| Learning rate | $5 \times 10^{-5}$ |
| Batch size efectivo | 16 |
| Max length (tokens) | 512 |
| Epocas | 10 |
| Dropout | 0.1 |
| Weight decay (L2) | 0.005 |
| Optimizador | AdamW |
| Capas congeladas | Ninguna (fine-tuning completo) |
| Seed | 10 |

### Diferencias respecto al NB-10 (baseline)

| Parametro | NB-10 (baseline) | NB-10-B (este) |
|---|---|---|
| MAX_LENGTH | 128 | **512** |
| EPOCHS | 3 | **10** |
| LR | 2e-5 | **5e-5** |
| weight_decay | 0.01 | **0.005** |
| dropout | 0.1 (default) | **0.1 (explicito)** |
| SEED | 42 | **10** |
| Batch fisico / acumulacion | 16 / 1 | **8 / 2** (= 16 efectivo) |
| EarlyStopping patience | 2 | **3** |

> **Batch size**: los investigadores usaron batch=16 con MAX_LENGTH=512.
> En T4 (15GB VRAM) eso es arriesgado. Usamos batch fisico=8 con
> `gradient_accumulation_steps=2` para lograr el **mismo batch efectivo de 16**
> sin riesgo de OOM. Matematicamente equivalente.

### Sistema de reanudacion
Mismo protocolo que NB-10: checkpoint JSON + modelos por fold en Drive.

---
### Instrucciones
1. **Runtime > Change runtime type > T4 GPU**.
2. Sube `train_val_set.csv` a: `MyDrive/ProyectoFinal_ML/data/`
3. Ejecuta en orden. Para reanudar: ejecuta todo de nuevo.

## Celda 0 — Verificacion del Hardware (GPU T4)

In [ ]:
!nvidia-smi

import torch
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "=" * 55)
print("  VERIFICACION DE HARDWARE")
print("=" * 55)
print(f"  Dispositivo activo : {device}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"  GPU                : {props.name}")
    print(f"  VRAM total         : {vram_gb:.1f} GB")
    torch.cuda.empty_cache()
    print("  Cache GPU          : limpiada")
else:
    raise RuntimeError(
        "GPU no detectada. Ve a Runtime > Change runtime type > T4 GPU."
    )
print("=" * 55)

## Celda 1 — Instalacion de Dependencias y Parche torchvision

In [ ]:
!pip install transformers datasets accelerate memory_profiler imbalanced-learn -q

# Parche para evitar ImportError de torchvision.io.VideoReader en Colab
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

import transformers, datasets, memory_profiler, imblearn

print(f"transformers    : {transformers.__version__}")
print(f"datasets        : {datasets.__version__}")
print(f"memory_profiler : {memory_profiler.__version__}")
print(f"imbalanced-learn: {imblearn.__version__}")
print(f"torchvision fix : datasets.config.TORCHVISION_AVAILABLE = False")
print("Dependencias listas.")

## Celda 2 — Montaje de Drive y Carga del Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd

# --- CONFIGURACION ---
BASE_PATH        = '/content/drive/MyDrive/ProyectoFinal_ML/data'
TRAIN_PATH       = os.path.join(BASE_PATH, 'train_val_set.csv')
# Archivos de estado SEPARADOS del NB-10 para no sobreescribir
ESTADO_PATH      = os.path.join(BASE_PATH, 'beto_mejorado_estado.json')
CHECKPOINTS_PATH = os.path.join(BASE_PATH, 'beto_mejorado_checkpoints')
# ---------------------

os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"No se encontro el archivo:\n  {TRAIN_PATH}\n"
        "Sube 'train_val_set.csv' a la carpeta indicada en Drive."
    )

df = pd.read_csv(TRAIN_PATH, sep=';')
df['texto_beto'] = df['texto_beto'].fillna('')

print("Dataset cargado.")
print(f"  Total de registros : {len(df):,}")
print(f"  Distribucion original:")
print(df['label'].value_counts().to_string())
print(f"\n  Estado checkpoints : {ESTADO_PATH}")
print(f"  Carpeta modelos    : {CHECKPOINTS_PATH}")

## Celda 3 — Importaciones y Configuracion (Hiperparametros de Investigacion)

In [ ]:
import time
import warnings
import gc
import json
import shutil
import numpy as np

from memory_profiler import memory_usage
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from imblearn.under_sampling import RandomUnderSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

warnings.filterwarnings('ignore')

# ===================================================================
#  HIPERPARAMETROS DE INVESTIGACION (F1 ~0.93 reportado)
# ===================================================================
MODELO_BETO = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LENGTH  = 512     # 512 tokens: contexto completo de BERT
BATCH_SIZE  = 8       # batch fisico (T4 seguro con 512 tokens)
GRAD_ACCUM  = 2       # gradient accumulation -> batch efectivo = 8*2 = 16
EPOCHS      = 10      # 10 epocas (vs 3 en NB-10 baseline)
LR          = 5e-5    # 5e-5 (vs 2e-5 en baseline)
WEIGHT_DECAY = 0.005  # L2 regularizacion (vs 0.01 en baseline)
DROPOUT     = 0.1     # dropout explicito en config del modelo
K_FOLDS     = 5
SEED        = 10      # seed del paper (vs 42 en baseline)
PATIENCE    = 3       # early stopping con mas paciencia (10 epocas lo amerita)
# ===================================================================

print("  CONFIGURACION — Hiperparametros de Investigacion")
print("=" * 55)
print(f"  Modelo        : {MODELO_BETO}")
print(f"  Max tokens    : {MAX_LENGTH}")
print(f"  Batch fisico  : {BATCH_SIZE}")
print(f"  Grad accum    : {GRAD_ACCUM} (batch efectivo = {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Epocas        : {EPOCHS}")
print(f"  Learning rate : {LR}")
print(f"  Weight decay  : {WEIGHT_DECAY}")
print(f"  Dropout       : {DROPOUT}")
print(f"  K-Folds       : {K_FOLDS}")
print(f"  Seed          : {SEED}")
print(f"  EarlyStopping : patience={PATIENCE}")
print("=" * 55)

## Celda 4 — Funciones de Estado (Guardar / Cargar / Reset)

In [ ]:
def guardar_estado(f1_lista, vram_lista, tiempos_lista):
    estado = {
        'folds_completados' : len(f1_lista),
        'f1_por_fold'       : f1_lista,
        'vram_por_fold'     : vram_lista,
        'tiempos_por_fold_s': tiempos_lista,
    }
    with open(ESTADO_PATH, 'w', encoding='utf-8') as fh:
        json.dump(estado, fh, indent=2)
    print(f"  [CHECKPOINT] Estado guardado: {len(f1_lista)}/{K_FOLDS} folds completos.")


def cargar_estado():
    if os.path.exists(ESTADO_PATH):
        with open(ESTADO_PATH, 'r', encoding='utf-8') as fh:
            estado = json.load(fh)
        n = estado['folds_completados']
        print(f"\n  [REANUDACION] Estado previo encontrado: {n}/{K_FOLDS} folds ya completos.")
        for i, f1 in enumerate(estado['f1_por_fold']):
            t_min = estado['tiempos_por_fold_s'][i] / 60
            print(f"    Fold {i+1}: F1={f1:.4f} | Tiempo={t_min:.1f} min")
        return (
            estado['f1_por_fold'],
            estado['vram_por_fold'],
            estado['tiempos_por_fold_s'],
            n
        )
    print("  [INICIO FRESCO] No se encontro estado previo. Empezando desde el Fold 1.")
    return [], [], [], 0


def resetear_estado():
    """ADVERTENCIA: borra todo el progreso guardado."""
    if os.path.exists(ESTADO_PATH):
        os.remove(ESTADO_PATH)
    if os.path.exists(CHECKPOINTS_PATH):
        shutil.rmtree(CHECKPOINTS_PATH)
        os.makedirs(CHECKPOINTS_PATH, exist_ok=True)
    print("Estado y checkpoints eliminados. Listo para empezar desde cero.")


# --- Descomenta SOLO si quieres empezar desde cero ---
# resetear_estado()

print("Funciones de estado definidas.")

## Celda 5 — Tokenizador y Funciones Auxiliares

In [ ]:
print(f"Descargando tokenizador: {MODELO_BETO}")
tokenizer = AutoTokenizer.from_pretrained(MODELO_BETO)
print("Tokenizador listo.")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1_macro': f1_score(labels, preds, average='macro')}


def construir_hf_dataset(textos, etiquetas):
    """Tokeniza con MAX_LENGTH=512 y empaqueta en Dataset HuggingFace."""
    raw = Dataset.from_dict({'texto': list(textos), 'labels': list(etiquetas)})

    def tokenize_fn(examples):
        return tokenizer(
            examples['texto'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH   # 512 tokens
        )

    tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['texto'])
    tokenized.set_format('torch')
    return tokenized


def cargar_beto_con_dropout():
    """
    Carga BETO con el dropout especificado explicitamente en la config.
    Los investigadores usaron dropout=0.1 (que es el default de BERT,
    pero lo fijamos explicitamente para reproducibilidad).
    Ninguna capa se congela: fine-tuning completo.
    """
    config = AutoConfig.from_pretrained(
        MODELO_BETO,
        num_labels=2,
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BETO,
        config=config,
        ignore_mismatched_sizes=True
    )
    # Verificar que NO hay capas congeladas
    n_total  = sum(1 for _ in model.parameters())
    n_train  = sum(1 for p in model.parameters() if p.requires_grad)
    print(f"  Parametros: {n_total} total, {n_train} entrenables (0 congelados)")
    return model


print(f"Funciones definidas. MAX_LENGTH={MAX_LENGTH}, DROPOUT={DROPOUT}")

## Celda 6 — Bucle K-Fold con Checkpoints Reanudables

### Flujo por fold
```
Cargar estado (Drive) --> saltar folds ya completos
      |
      v (solo folds pendientes)
Split estratificado
  |-- TRAIN (80%) --> RandomUnderSampler --> TRAIN BALANCEADO
  |                                              |
  |                   Tokenizar (512 tokens) <---'
  |                         |
  |                   BETO (dropout=0.1, todas capas descongeladas)
  |                   AdamW (lr=5e-5, wd=0.005)
  |                   10 epocas, fp16, EarlyStopping(patience=3)
  |                         |
  `-- VAL (20%) --> sin modificar --> evaluacion F1-Score Macro
      |
      v
Guardar modelo fold en Drive --> Guardar estado JSON
```

> **Tiempo estimado**: 10 epocas x 5 folds con 512 tokens ~ 2-4 horas en T4.
> El sistema de checkpoints permite reanudar si la sesion se corta.

In [ ]:
textos    = df['texto_beto'].tolist()
etiquetas = df['label'].tolist()

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
todos_los_splits = list(skf.split(textos, etiquetas))

# --- Cargar progreso previo ---
f1_por_fold, vram_por_fold, tiempos_por_fold, fold_inicio = cargar_estado()

if fold_inicio >= K_FOLDS:
    print("\nTodos los folds ya estan completos. Salta a la Celda 7.")
else:
    print(f"\nReanudando desde el Fold {fold_inicio + 1} de {K_FOLDS}.")


def ejecutar_folds_pendientes():
    for fold_idx in range(fold_inicio, K_FOLDS):
        train_idx, val_idx = todos_los_splits[fold_idx]

        print(f"\n{'='*60}")
        print(f"  FOLD {fold_idx + 1} / {K_FOLDS}")
        print(f"{'='*60}")

        t_fold_inicio = time.time()

        # --- Particion cruda ---
        textos_train_raw = [textos[i] for i in train_idx]
        labels_train_raw = [etiquetas[i] for i in train_idx]
        textos_val       = [textos[i] for i in val_idx]
        labels_val       = [etiquetas[i] for i in val_idx]

        print(f"  Train crudo: {len(textos_train_raw):,} | Val: {len(textos_val):,}")

        # --- RandomUnderSampler SOLO sobre train (equidad benchmark) ---
        rus = RandomUnderSampler(random_state=SEED)
        idx_array = np.array(range(len(textos_train_raw))).reshape(-1, 1)
        idx_res, labels_train = rus.fit_resample(idx_array, labels_train_raw)
        textos_train = [textos_train_raw[i[0]] for i in idx_res]

        print(f"  Train balanceado: {len(textos_train):,}")
        print(f"  Dist.: {dict(zip(*np.unique(labels_train, return_counts=True)))}")

        # --- Tokenizacion (512 tokens) ---
        print("  Tokenizando (MAX_LENGTH=512)...")
        ds_train = construir_hf_dataset(textos_train, labels_train)
        ds_val   = construir_hf_dataset(textos_val,   labels_val)

        # --- Modelo BETO fresco con dropout explicito ---
        print("  Cargando BETO en GPU (dropout=0.1, fine-tuning completo)...")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        model_fold = cargar_beto_con_dropout()
        model_fold.to(device)

        output_dir_local = f'/content/beto_mejorado_fold_{fold_idx + 1}'

        training_args = TrainingArguments(
            output_dir=output_dir_local,
            num_train_epochs=EPOCHS,                              # 10 epocas
            per_device_train_batch_size=BATCH_SIZE,               # 8 fisico
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,               # x2 = batch efectivo 16
            learning_rate=LR,                                     # 5e-5
            warmup_ratio=0.1,
            weight_decay=WEIGHT_DECAY,                            # 0.005
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            logging_steps=50,
            fp16=True,
            dataloader_num_workers=0,
            report_to='none',
            seed=SEED                                             # seed=10
        )

        trainer = Trainer(
            model=model_fold,
            args=training_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
        )

        print(f"  Entrenando ({EPOCHS} epocas, lr={LR}, wd={WEIGHT_DECAY})...")
        trainer.train()

        # --- Metricas del fold ---
        metricas = trainer.evaluate()
        f1_fold  = metricas['eval_f1_macro']
        vram_gb  = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
        t_fold_s = time.time() - t_fold_inicio

        f1_por_fold.append(f1_fold)
        vram_por_fold.append(vram_gb)
        tiempos_por_fold.append(t_fold_s)

        print(f"  F1-Score Fold {fold_idx+1}: {f1_fold:.4f}")
        print(f"  VRAM pico         : {vram_gb:.2f} GB")
        print(f"  Tiempo fold       : {t_fold_s/60:.1f} min")

        # --- Guardar modelo en Drive ---
        fold_drive_path = os.path.join(CHECKPOINTS_PATH, f'fold_{fold_idx + 1}')
        os.makedirs(fold_drive_path, exist_ok=True)
        model_fold.save_pretrained(fold_drive_path)
        tokenizer.save_pretrained(fold_drive_path)
        print(f"  Modelo guardado: {fold_drive_path}")

        # --- Guardar estado JSON ---
        guardar_estado(f1_por_fold, vram_por_fold, tiempos_por_fold)

        # --- Limpieza ---
        del model_fold, trainer, ds_train, ds_val
        if os.path.exists(output_dir_local):
            shutil.rmtree(output_dir_local)
        torch.cuda.empty_cache()
        gc.collect()


# --- EJECUCION con medicion de RAM ---
if fold_inicio < K_FOLDS:
    print(f"\nIniciando entrenamiento BETO MEJORADO...")
    print(f"Corpus: {len(textos):,} | Epocas: {EPOCHS} | Batch efectivo: {BATCH_SIZE*GRAD_ACCUM}")
    print(f"lr: {LR} | wd: {WEIGHT_DECAY} | max_len: {MAX_LENGTH} | seed: {SEED}")

    muestras_ram = memory_usage(
        (ejecutar_folds_pendientes, [], {}),
        interval=1.0,
        include_children=True,
        max_usage=False,
        retval=False
    )

    ram_sesion_gb = max(muestras_ram) / 1024
    print(f"\nFolds completados. RAM pico esta sesion: {ram_sesion_gb:.2f} GB")
else:
    ram_sesion_gb = 0.0

print("\nCelda 6 finalizada. Continua a la Celda 7.")

## Celda 7 — Resultados Finales

> Lee directamente del archivo de estado en Drive.

In [ ]:
f1_final_lista, vram_final_lista, tiempos_final_lista, n_completos = cargar_estado()

if n_completos < K_FOLDS:
    print(f"\nAun faltan {K_FOLDS - n_completos} fold(s). Ejecuta la Celda 6 para continuar.")
else:
    tiempo_total_min = sum(tiempos_final_lista) / 60
    f1_final         = float(np.mean(f1_final_lista))
    vram_pico_gb     = max(vram_final_lista)
    ram_nota         = ram_sesion_gb if 'ram_sesion_gb' in dir() else float('nan')

    print("\n" + "=" * 75)
    print("  RESULTADOS FINALES — BETO MEJORADO K=5 (Hiperparametros de Investigacion)")
    print("  Entorno: Google Colab — T4 GPU Runtime")
    print("=" * 75)
    if not np.isnan(ram_nota):
        print(f"  BETO Mejorado: F1-Score: {f1_final:.2f} | Tiempo: {tiempo_total_min:.1f} min | RAM: {ram_nota:.2f} GB | VRAM: {vram_pico_gb:.2f} GB")
    else:
        print(f"  BETO Mejorado: F1-Score: {f1_final:.2f} | Tiempo: {tiempo_total_min:.1f} min | VRAM: {vram_pico_gb:.2f} GB")
    print("=" * 75)

    print("\n  Detalle por fold:")
    for i in range(K_FOLDS):
        t_min = tiempos_final_lista[i] / 60
        print(f"    Fold {i+1}: F1={f1_final_lista[i]:.4f} | "
              f"Tiempo={t_min:.1f} min | VRAM={vram_final_lista[i]:.2f} GB")
    print(f"  Promedio F1 : {f1_final:.4f}")
    print(f"  Desv. std   : {float(np.std(f1_final_lista)):.4f}")

    print("\n  Configuracion usada:")
    print(f"    lr={LR} | wd={WEIGHT_DECAY} | dropout={DROPOUT}")
    print(f"    max_len={MAX_LENGTH} | epochs={EPOCHS} | batch_eff={BATCH_SIZE*GRAD_ACCUM}")
    print(f"    seed={SEED} | early_stop=patience({PATIENCE})")

    resumen = pd.DataFrame({
        'Modelo':             ['BETO Mejorado (config investigacion)'],
        'F1-Score Macro':     [round(f1_final, 4)],
        'Desv. Std':          [round(float(np.std(f1_final_lista)), 4)],
        'Tiempo total (min)': [round(tiempo_total_min, 1)],
        'VRAM Pico (GB)':     [round(vram_pico_gb, 2)],
        'K-Folds':            [K_FOLDS],
        'Epocas/Fold':        [EPOCHS],
        'Batch Efectivo':     [BATCH_SIZE * GRAD_ACCUM],
        'Max Length':         [MAX_LENGTH],
        'LR':                 [LR],
        'Weight Decay':       [WEIGHT_DECAY],
        'Dropout':            [DROPOUT],
        'Seed':               [SEED],
        'Balanceo train':     ['RandomUnderSampler']
    })
    display(resumen.set_index('Modelo'))

## Celda 8 — Entrenamiento Final y Exportacion a Drive

> Ejecuta **una sola vez**, cuando todos los K=5 folds esten completos.

In [ ]:
_, _, _, n_check = cargar_estado()

if n_check < K_FOLDS:
    raise RuntimeError(
        f"Solo hay {n_check}/{K_FOLDS} folds completos.\n"
        "Completa todos los folds antes de ejecutar esta celda."
    )

print("Entrenamiento final (corpus completo balanceado, config investigacion)...")

rus_final = RandomUnderSampler(random_state=SEED)
idx_todos = np.array(range(len(textos))).reshape(-1, 1)
idx_bal, labels_bal = rus_final.fit_resample(idx_todos, etiquetas)
textos_bal = [textos[i[0]] for i in idx_bal]

print(f"  Corpus original  : {len(textos):,}")
print(f"  Corpus balanceado: {len(textos_bal):,}")
print(f"  Distribucion     : {dict(zip(*np.unique(labels_bal, return_counts=True)))}")

ds_completo = construir_hf_dataset(textos_bal, labels_bal)

torch.cuda.empty_cache()
model_final = cargar_beto_con_dropout()
model_final.to(device)

args_final = TrainingArguments(
    output_dir='/content/beto_mejorado_final',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=WEIGHT_DECAY,
    logging_steps=100,
    save_strategy='no',
    fp16=True,
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED
)

trainer_final = Trainer(
    model=model_final,
    args=args_final,
    train_dataset=ds_completo,
    compute_metrics=compute_metrics
)

trainer_final.train()
print("Entrenamiento final completado.")

DRIVE_MODEL_PATH = os.path.join(BASE_PATH, 'beto_mejorado_final_model')
os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)
model_final.save_pretrained(DRIVE_MODEL_PATH)
tokenizer.save_pretrained(DRIVE_MODEL_PATH)
print(f"Modelo final exportado: {DRIVE_MODEL_PATH}")

PATH_CSV = os.path.join(BASE_PATH, 'resultados_beto_mejorado_kfold.csv')
resumen.to_csv(PATH_CSV, index=False, sep=';')
print(f"Resultados guardados  : {PATH_CSV}")
print("\nTodo exportado a Google Drive correctamente.")